# CultureTechLens — Black Cultural Knowledge Graph: Starter Notebook

**Culture, Clearly Seen.** · Black Cultural Intelligence · Source: CultureTechLens

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/culturetechlens/culturetechlens/blob/main/notebooks/ctl-graph-starter.ipynb)

This notebook loads the published CultureTechLens knowledge-graph slice — **3,488 evidence-graded records and 2,162 relationships** — and answers real questions with it: records by decade, the places that anchor the graph, and who was honored at the Soul Train Awards, category by category.

**License:** CC BY 4.0 · **Cite as:** CultureTechLens, *Black Cultural Knowledge Graph* — concept DOI [10.5281/zenodo.23403147](https://doi.org/10.5281/zenodo.23403147). Canonical counts: `counts.json` in the flagship repository (generated, never hand-edited, verified in CI).

In [ ]:
import json, collections
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/culturetechlens/culturetechlens/main"
entities = pd.read_csv(f"{BASE}/knowledge-graph/entities.csv")
edges = pd.read_csv(f"{BASE}/knowledge-graph/edges.csv")
props = entities["props_json"].apply(json.loads)
entities["verification"] = props.apply(lambda p: (p.get("verification") or {}).get("status"))
print(f"{len(entities)} entities · {len(edges)} relationships")
entities["verification"].value_counts()

## Records by type — the shape of the archive

Eight record types: Person, Work, Place, Event, Institution, Outlet, Topic, List.

In [ ]:
entities["node_type"].value_counts().plot.barh(color="#B3282D")
plt.title("Records by type — CultureTechLens stamped slice")
plt.xlabel("records"); plt.gca().invert_yaxis(); plt.show()

## Records by decade

733 records carry a `year` or `debut_year`. The archive's center of gravity is the 1970s–2010s — the soul, funk, hip-hop, and Black television eras.

In [ ]:
def get_year(p):
    y = p.get("year") or p.get("debut_year")
    return y if isinstance(y, int) and 1800 < y < 2030 else None
years = props.apply(get_year).dropna().astype(int)
(years // 10 * 10).value_counts().sort_index().plot.bar(color="#C9A227")
plt.title("Dated records by decade"); plt.xlabel("decade"); plt.ylabel("records"); plt.show()

## The places that anchor the graph

`LOCATED_IN` edges point at the graph's geography. One city is the hub.

In [ ]:
byid = entities.set_index("kg_id")["name"]
loc = edges[edges.edge_type == "LOCATED_IN"]["to_id"].value_counts().head(10)
loc.index = [byid.get(i, i) for i in loc.index]
loc.plot.barh(color="#1E7A46"); plt.title("Records located in — top places"); plt.gca().invert_yaxis(); plt.show()
print("Chicago alone anchors", int(loc.get("Chicago", 0)), "located-in records in this slice.")

## Worked query — who was honored at the Soul Train Awards, and in which category?

`HONORED_BY` edges carry the category in their notes — 658 of them. A journalist can rebuild an awards table from the graph alone.

In [ ]:
awards = edges[edges.edge_type == "HONORED_BY"].copy()
awards["honoree"] = awards["from_id"].map(byid)
awards["event"] = awards["to_id"].map(byid)
sample = awards[awards["event"].str.contains("Soul Train", na=False)].head(10)
for _, r in sample.iterrows():
    print(f"{r['honoree']} — {r['event']} — {r['notes'][:80]}")

## Export the overview figure

One figure that carries the whole story — the same hub view published with the dataset.

In [ ]:
import math
deg = collections.Counter()
for _, e in edges.iterrows():
    deg[e.from_id] += 1; deg[e.to_id] += 1
center = deg.most_common(1)[0][0]
nbrs = edges[(edges.from_id == center) | (edges.to_id == center)]
others = list({*nbrs.from_id, *nbrs.to_id} - {center})[:36]
pos = {center: (0, 0)}
for j, k in enumerate(others):
    a = 2 * math.pi * j / len(others)
    pos[k] = (math.cos(a), math.sin(a))
plt.figure(figsize=(8, 8))
for _, e in nbrs.iterrows():
    if e.from_id in pos and e.to_id in pos:
        (x1, y1), (x2, y2) = pos[e.from_id], pos[e.to_id]
        plt.plot([x1, x2], [y1, y2], color="#C9A227", alpha=0.35, lw=0.7)
for k, (x, y) in pos.items():
    plt.scatter([x], [y], s=900 if k == center else 90, color="#B3282D" if k == center else "#16150F")
    if k != center:
        plt.annotate(byid.get(k, k)[:22], (x, y), fontsize=6, ha="center", xytext=(0, 7), textcoords="offset_points")
plt.title(f"The hub: {byid[center]} — {deg[center]} relationships (stamped slice)")
plt.axis("off"); plt.savefig("graph-overview.png", dpi=150, bbox_inches="tight"); plt.show()
print("saved graph-overview.png")

## Where to go next

- **Starter queries for teachers and journalists:** `docs/starter-queries.md` in this repository.
- **Interactive graph:** the GitHub Pages site for this repo — `docs/graph.html`.
- **Found an error?** CultureTechLens corrects in public — open the *Report an error* form under Issues. Verified corrections are published in the corrections log at culturetechlens.com/about/corrections/.
- **A record that should exist but doesn't?** The *Missing record* form is the door.

CULTURETECHLENS · "Culture, Clearly Seen." · Black Cultural Intelligence · Source: CultureTechLens